# Distilling Qwen3-Embedding-4B into RoBERTa

**Question (Further Work):** can RoBERTa-base learn what the fine-tuned Qwen3-Embedding-4B knows, and close the gap between them?

| model | test accuracy | weights |
|---|---|---|
| RoBERTa-base, all weights fine-tuned (`sweep.py`, mean of 3 seeds) | 73.13 | 125M |
| Qwen3-Embedding-4B, LoRA fine-tuned (`LoRA Qwen3-Embedding-4B.ipynb`, one run) | 78.68 | 4.0B |

The gap is 5.55 points, 411 test posts. Qwen is better, but it's 32× bigger, so each prediction costs roughly 32× the computation. **Knowledge distillation** trains a small model (the *student*) to imitate a big one (the *teacher*). If RoBERTa can learn what Qwen knows, we'd get Qwen's accuracy at RoBERTa's cost.

This is `Fine-tuned Roberta.ipynb` with one change: **the loss.** RoBERTa still learns from the true labels, but it also learns to match Qwen's scores for all 20 classes on every training post. Same data, thread-grouped dev split, model, training recipe and test set.

**Hypothesis:** distilled RoBERTa closes some or all of the gap.

**Predict before running:** where between 73.13 and 78.68 will it land?

## Knowledge distillation

**Read first:** Hinton, Vinyals & Dean 2015, "Distilling the Knowledge in a Neural Network", Sections 1 and 2 (about 3 pages): https://arxiv.org/abs/1503.02531. Neither the lectures nor J&M cover distillation, but it's built from pieces they do cover: softmax, cross-entropy, temperature and KL divergence.

**Hard and soft labels.** A post's label names one class: a *hard* label. The teacher's softmax gives all 20 classes a probability: a *soft* label. For a post from `comp.sys.mac.hardware`, say:
- hard label: `mac.hardware` 1, the other 19 classes 0;
- teacher: `mac.hardware` 0.90, `ibm.pc.hardware` 0.07, `sci.electronics` 0.02, ..., `rec.sport.hockey` 0.0001.

The small numbers say what the post resembles: a PC-hardware post much more than a hockey one. Hinton calls this *dark knowledge*, and the hard label throws it away. A soft label gives the student 20 numbers per post to learn from instead of one: "When the soft targets have high entropy, they provide much more information per training case than hard targets and much less variance in the gradient between training cases" (Section 1).

**Temperature.** A trained teacher is usually confident, so its runner-ups get probabilities like 0.0001, too small to teach much. A temperature $T$ softens its distribution:

$$p_c = \frac{\exp(z_c / T)}{\sum_{c'} \exp(z_{c'} / T)}$$

$z$ are the logits, the 20 scores that go into the softmax. $T = 1$ is the ordinary softmax. $T > 1$ shrinks the gaps between the logits, so the distribution flattens and the runner-ups show. It's the temperature of J&M Section 7.6.3, "Temperature sampling" (pp. 198–200 in the current draft, Figure 7.21), used the other way around: generation uses $\tau < 1$ to sharpen the distribution toward likely words, while distillation uses $T > 1$ to flatten it and expose the unlikely classes.

**The loss** adds two parts:

$$L = \alpha \cdot \text{CE}\big(y, \text{softmax}(z_s)\big) + (1 - \alpha) \cdot T^2 \cdot \text{KL}\big(\text{softmax}(z_t / T) \,\big\|\, \text{softmax}(z_s / T)\big)$$

- $z_s$ are the student's logits, $z_t$ the teacher's, and $y$ the true label.
- **The hard part** is the plain fine-tune's loss: cross-entropy with the true label, at $T = 1$.
- **The soft part** compares the teacher's and the student's distributions, both at temperature $T$, with the **KL divergence** (J&M eq. 22.38, p. 513): $\text{KL}(p \,\|\, q) = \sum_c p_c \log \frac{p_c}{q_c}$. It's 0 when the two distributions are equal, and positive otherwise.
- **KL and cross-entropy differ by a constant:** $\text{KL}(p \,\|\, q) = -\sum_c p_c \log q_c + \sum_c p_c \log p_c$, the cross-entropy with soft targets minus the teacher's entropy. The teacher's entropy doesn't depend on the student, so the two losses have the same gradients. Hinton writes the loss with cross-entropy; KL has the advantage of being exactly 0 when the student matches the teacher.
- **Why $T^2$:** the soft part's gradients shrink like $1/T^2$, so Hinton multiplies it by $T^2$ "when using both hard and soft targets" (Section 2). That keeps the balance between the two parts about the same whatever $T$ is.
- **$\alpha$ weights the true labels.** Hinton got his best results with a considerably lower weight on the hard part. We start from $\alpha = 0.5$, an equal split. $\alpha = 1$ is the plain fine-tune.
- **The teacher never changes.** Its logits are computed once and saved. To the student they're fixed targets, like the labels.

**At high temperature, distillation becomes matching the logits:** minimizing $\frac{1}{2}(z_s - z_t)^2$ for each class (Hinton, Section 2.1). Some later work, like Tang et al. below, distills that way directly.

**Further reading (optional):**
- Sanh et al. 2019, DistilBERT: https://arxiv.org/abs/1910.01108. Soft labels (plus two other losses) while *pre-training* a 6-layer BERT on BERT's own pre-training text. It kept 97% of BERT's score on GLUE with 40% fewer weights, and runs 60% faster.
- Tang et al. 2019, "Distilling Task-Specific Knowledge from BERT into Simple Neural Networks": https://arxiv.org/abs/1903.12136. Distillation for one task, as here, from a fine-tuned BERT into a single-layer BiLSTM, by matching logits. To give the student more to learn from, they also generated many new training inputs by data augmentation, which the teacher then labeled.

## Will it work here?

**Why it could:**
- **The soft labels carry what Qwen knows about how the 20 groups relate:** Mac vs. PC hardware, the three religion groups, politics vs. guns. Qwen learned this in pre-training, in its embedding training (about 170M text pairs; `fine-tuning-results.md`, conjecture 1) and in its fine-tuning. These are the groups RoBERTa's errors fall between (`Lessons.md`, Error analysis).
- **Soft labels also regularize.** In Hinton's speech experiment, a model trained on 3% of the data overfit with hard labels (44.5% test frame accuracy) but reached 57.0% with soft labels, close to the 58.9% of a model trained on all of it (Section 6, Table 5). Fine-tuned RoBERTa overfits too: its training loss falls to 0.06 while its dev loss climbs from 0.82 to 1.46.

**Why it might not:**
- **The student sees no new posts.** The *transfer set*, the posts on which the student imitates the teacher, is the same 9,987 training posts it learned from before. Qwen's knowledge can reach it only through 20 numbers per post. Large gains in the literature usually come with far more transfer inputs: DistilBERT's whole pre-training corpus, Tang et al.'s augmented data.
- **The teacher trained on these very posts.** After its 3 epochs, Qwen's training loss was 0.146 against a dev loss of 0.897 (`LoRA Qwen3-Embedding-4B.ipynb`), so it's far more confident on these posts than on new ones. On a post it's sure of, its soft label is almost the hard label, with little dark knowledge left. The temperature is what brings it back; the cells below measure how much.
- **The capacity gap.** Qwen has 32× RoBERTa's weights. Cho & Hariharan 2019 ("On the Efficacy of Knowledge Distillation", ICCV) found that more accurate teachers don't always make better teachers: a much smaller student can't follow a much bigger one.
- **A gain alone wouldn't prove that Qwen's knowledge moved.** Yuan et al. 2020 ("Revisiting Knowledge Distillation via Label Smoothing Regularization", CVPR) found that even a weak teacher, worse than the student, can help it. Part of what soft labels do is *label smoothing*: training toward a little less than 100% on the true class, a regularizer that needs no teacher. Telling the two apart needs a control run ("Next steps", at the end).

So the result could be anything from "no gain" to "most of the gap". The measure is **the fraction of the gap closed**: (distilled − 73.13) / (78.68 − 73.13).

## How to run it

The full fine-tune's learning rate was chosen from 3 values on dev before 3 seeds were scored on test. Distillation gets the same effort, spent on its own new setting, the temperature (as in `LoRA Roberta.ipynb`):
1. **The first run computes the teacher's logits** and saves them to `qwen3-embedding-4b-lora-logits.npz`: about 9 minutes, with about 9.5 GB of GPU memory free (close other GPU programs first). Later runs load the file in a second.
2. **Choose the temperature on dev:** run with `SEED = 423`, `ALPHA = 0.5` and `TEMPERATURE` = 1, 2 and 4, keeping `SCORE_TEST = False`. Each run appends its dev scores to `../sweeps/sweep.jsonl` and saves its dev and test probabilities in `../sweeps/probs/`, without scoring test.
3. `~/venvs/cs423/bin/python ../sweeps/summarize.py` lists every run's dev accuracy by epoch. Take the temperature with the highest best-epoch dev accuracy (on a tie, 2). If 4 wins, run 8 too, so that the best value isn't at the edge of the grid (`Lessons.md`, Search mechanics). Decide before step 4.
4. Run the chosen temperature again with `SEED = 1` and `SEED = 2`, the other seeds `sweep.py` used.
5. `~/venvs/cs423/bin/python ../sweeps/summarize.py --final kd_T2_a0.5 --select acc` (with the chosen run's name) scores the 3 seeds and their ensemble on test. The last code cell of this notebook then compares them with the plain fine-tune and with the teacher.

**Decision rule, fixed before any run.** Gain = distillation's 3-seed mean minus the plain fine-tune's (73.13):
- **under 1 point:** no clear effect. The plain fine-tune's 3 `sweep.py` seeds had a standard deviation of 0.31, and all 4 of its RoBERTa-recipe runs 0.92.
- **1 point or more, and McNemar p < 0.05** between the two 3-seed ensembles: distillation helps.
- **1 point or more, but p ≥ 0.05:** inconclusive; more seeds.
- **Whichever it is, report the fraction of the gap closed.**

**Time:** a run takes about 8 minutes, like the full fine-tune: the teacher adds nothing to a training step, since its logits are already computed. The first run adds the teacher's 9 minutes. All 5 runs: about 50 minutes.

In [1]:
## Settings for this run (see "How to run it")
TEACHER = "frozen_mlp_oof_calibrated"   ## "lora" (Part 1), "frozen_lr_oof" (Part 1b), "label_smoothing", "frozen_lr_insample" (1c), "frozen_mlp_oof", "frozen_mlp_oof_calibrated" (1d)
TEMPERATURE = 1     ## T: 1, 2 and 4 while choosing it on dev (the frozen_lr_oof teacher is already calibrated: 1)
ALPHA = 0.5         ## the weight on the true labels; 1 - ALPHA goes to imitating the teacher. ALPHA = 1: the plain fine-tune
SEED = 2          ## 423 while choosing T, then 1 and 2
SCORE_TEST = True  ## True prints this run's test accuracy at the end
## f"{x:g}" writes a number in its shortest form: 2 -> "2", 0.5 -> "0.5"
NAME = f"kd{'' if TEACHER == 'lora' else '_' + TEACHER}_T{TEMPERATURE:g}_a{ALPHA:g}"   ## e.g. "kd_T2_a0.5" (LoRA teacher), "kd_frozen_lr_oof_T1_a0.5"
print(NAME, "seed", SEED)

kd_frozen_mlp_oof_calibrated_T1_a0.5 seed 2


## Data and dev split

As in the full fine-tune, whose comments explain the Python: headers, footers and quotes removed, empty posts dropped, and the same 1,109 thread-grouped dev posts. The teacher was trained on exactly these 9,987 training posts (same split, same seed), and has never seen the dev posts.

In [2]:
import re
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import StratifiedGroupKFold

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)      ## (11096,)
y_test = np.array(newsgroups_test.target)        ## (7370,)
n_classes = len(newsgroups_train.target_names)   ## 20
names = newsgroups_train.target_names            ## the 20 newsgroup names, in class-id order

## Dev set: 10% of the threads (a thread = the subject line without "Re:"), the same split as the other notebooks
with_headers = fetch_20newsgroups(subset='train')
subject_of = {f: re.search(r"^Subject:(.*)$", doc, re.M).group(1)
              for f, doc in zip(with_headers.filenames, with_headers.data)}
threads = [re.sub(r"^(re:\s*)+", "", subject_of[f].strip(), flags=re.IGNORECASE).lower()
           for f in newsgroups_train.filenames]
thread_ids = np.unique(threads, return_inverse=True)[1]
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)   ## 423, not SEED: the same dev set every run
train_idx, dev_idx = next(splitter.split(newsgroups_train.data, y_train, groups=thread_ids))
X_tr, y_tr = [newsgroups_train.data[i] for i in train_idx], y_train[train_idx]   ## 9,987 posts
X_dev, y_dev = [newsgroups_train.data[i] for i in dev_idx], y_train[dev_idx]     ## 1,109 posts
print(f"Training part: {len(X_tr):,} posts, dev: {len(X_dev):,} posts")

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


Training part: 9,987 posts, dev: 1,109 posts


## Part 1: the teacher's logits

The teacher is the LoRA-fine-tuned Qwen3-Embedding-4B of `LoRA Qwen3-Embedding-4B.ipynb`, which explains all of its code. That notebook saved only the weights it trained, LoRA's $A$'s and $B$'s and the head, to `qwen3-embedding-4b-lora.pt` (33M numbers, 132 MB). The other 4 billion weights are pre-trained Qwen's, which `from_pretrained` loads from the Hugging Face cache. So we rebuild the model exactly as that notebook built it, then load the file into it:
- **`load_state_dict(..., strict=False)`.** Loading normally checks that the file has a tensor for every weight in the model, and stops if one is missing. Here the 4 billion frozen weights are missing from the file on purpose, so that check is switched off, and we do our own: the file's tensor names must be exactly the model's trainable ones, with none left over. Without it, a misspelled name would silently leave a layer's $B$ at 0, or the head random.
- **The same input as in training:** the instruction in front, 512 Qwen tokens, the last token's vector, and posts grouped by length into pieces of at most 2,048 tokens, in bfloat16. Any difference would make it a slightly different model.
- **A check that it worked:** after its last epoch, the LoRA notebook scored 82.24 on these dev posts. The rebuilt teacher must score that again, give or take a post or two, since bfloat16 rounding can depend on which posts share a piece.
- **Logits, not probabilities,** because the softmax at temperature $T$ needs the logits, whatever $T$ we try.
- **Which posts:** the 9,987 training posts (what the student learns from), the 1,109 dev posts (for the check, and to measure how often the student agrees with the teacher), and the 7,370 test posts (only for the final comparison: the teacher's test score, 78.68, is already known, so nothing is chosen with it).
- **Memory:** about 9 GB, as in the LoRA notebook's evaluation. All of it is freed before the student is built.

About 9 minutes in all (measured: 9.0).

In [3]:
import time
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

device = "cuda"
TEACHER_MODEL = "Qwen/Qwen3-Embedding-4B"
LORA_FILE = "../lora/qwen3-embedding-4b-lora.pt"           ## the trained A's, B's and head, saved by the LoRA notebook
TEACHER_FILES = {"lora": "qwen3-embedding-4b-lora-logits.npz",           ## where this notebook saves each teacher's logits
                 "frozen_lr_oof": "qwen3-embedding-4b-frozen-lr-oof-logits.npz",
                 "frozen_lr_insample": "qwen3-embedding-4b-frozen-lr-insample-logits.npz",
                 "label_smoothing": "label-smoothing-logits.npz",
                 "frozen_mlp_oof": "qwen3-embedding-4b-frozen-mlp-oof-logits.npz",
                 "frozen_mlp_oof_calibrated": "qwen3-embedding-4b-frozen-mlp-oof-calibrated-logits.npz"}
TEACHER_FILE = Path(TEACHER_FILES[TEACHER])
INSTRUCTION = "Instruct: Classify the given Usenet newsgroup post by the newsgroup it was posted to\nQuery:"
QWEN_MAX_LENGTH = 512
TOKENS_PER_PIECE = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05   ## LoRA's r and alpha, as in the LoRA notebook (not the distillation's ALPHA)

## The LoRA notebook's classes, unchanged: a frozen nn.Linear plus (alpha / r) B A x, and Qwen with a head on the
## last token's vector
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Linear(base.in_features, r, bias=False, device=base.weight.device)   ## A: (r, d_in)
        self.B = nn.Linear(r, base.out_features, bias=False, device=base.weight.device)  ## B: (d_out, r)
        nn.init.zeros_(self.B.weight)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scale

class QwenClassifier(nn.Module):
    def __init__(self, qwen, n_classes):
        super().__init__()
        self.qwen = qwen
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(qwen.config.hidden_size, n_classes)   ## (20, 2560), (20,)

    def forward(self, input_ids, attention_mask):   ## both (N, L)
        hidden = self.qwen(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
        last = attention_mask.sum(dim=1) - 1                                  ## (N,): where each post's last token is
        vec = hidden[torch.arange(len(last), device=hidden.device), last]    ## (N, 2560)
        return self.head(self.dropout(vec))                                   ## (N, 20)

## The LoRA notebook's helpers. pieces: split a batch (positions in a list of posts) into pieces of at most
## TOKENS_PER_PIECE tokens. pad: a list of N token-id lists -> input_ids and attention_mask, both (N, L), padded on
## the right, in GPU memory.
def pieces(batch, lengths):
    per_piece = max(1, TOKENS_PER_PIECE // lengths[batch].max())
    return [batch[i:i + per_piece] for i in range(0, len(batch), per_piece)]

def pad(seqs, pad_id):
    L = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), L), pad_id)
    attention_mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s)
        attention_mask[i, :len(s)] = 1
    return input_ids.to(device), attention_mask.to(device)

## The teacher's logits for a list of tokenized posts, in their original order: the LoRA notebook's evaluate(), but
## returning the logits instead of scoring them. The posts go through sorted by length, 16 at a time, in pieces.
def teacher_logits(teacher, seqs, pad_id):
    lengths = np.array([len(s) for s in seqs])
    order = np.argsort(lengths)                   ## the posts' positions, shortest post first
    out = torch.empty((len(seqs), n_classes))     ## (number of posts, 20), in CPU memory, filled in below
    teacher.eval()                                ## dropout off
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for i in range(0, len(order), 16):
            for idx in pieces(order[i:i + 16], lengths):
                out[idx] = teacher(*pad([seqs[j] for j in idx], pad_id)).float().cpu()   ## rows idx get (N, 20)
    return out.numpy()

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Builds the teacher, loads its trained weights, checks them, and returns its logits for the training, dev and test
## posts: three numpy arrays, (9987, 20), (1109, 20) and (7370, 20). Everything it builds is local to the function, so
## once it returns, nothing refers to the teacher anymore and its memory can be freed (like a C++ object going out of
## scope).
def compute_teacher_logits():
    ## mem_get_info() asks the GPU driver for (free bytes, total bytes), counting every program's use, not just ours
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free: {free / 1e9:.1f} of {total / 1e9:.1f} GB")
    if free < 9.5e9:
        raise RuntimeError("The teacher needs about 9.5 GB of GPU memory: close other GPU programs and notebooks first")
    start = time.perf_counter()

    ## The LoRA notebook's tokenization: the instruction in front, at most 512 tokens, <|endoftext|> at the end
    qwen_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
    def tokenize(texts):   ## a function inside a function can use qwen_tokenizer, like a C++ lambda that captures it
        return qwen_tokenizer([INSTRUCTION + t for t in texts], truncation=True, max_length=QWEN_MAX_LENGTH)["input_ids"]

    ## Build the model exactly as the LoRA notebook did: frozen bfloat16 Qwen, a LoRALinear around each of the 7 matrices
    ## in each of the 36 layers (random A's, B = 0 for now), and a random head
    qwen = AutoModel.from_pretrained(TEACHER_MODEL, torch_dtype=torch.bfloat16).to(device)
    qwen.requires_grad_(False)
    for layer in qwen.layers:
        for module, proj_names in [(layer.self_attn, ["q_proj", "k_proj", "v_proj", "o_proj"]),
                                   (layer.mlp, ["gate_proj", "up_proj", "down_proj"])]:
            for name in proj_names:
                setattr(module, name, LoRALinear(getattr(module, name), LORA_R, LORA_ALPHA, LORA_DROPOUT))
    teacher = QwenClassifier(qwen, n_classes).to(device)

    ## Load the trained A's, B's and head over the random ones. torch.load returns the dict that was saved, name -> tensor:
    ## {"qwen.layers.0.self_attn.q_proj.A.weight": a (16, 2560) tensor, ..., "head.bias": a (20,) tensor}
    saved = torch.load(LORA_FILE)
    ## strict=False: the frozen weights aren't in the file, and that's expected. load_state_dict returns the names it
    ## couldn't match: missing_keys (in the model but not in the file) and unexpected_keys (in the file but not in the model).
    result = teacher.load_state_dict(saved, strict=False)
    ## Our own check instead: the file must hold exactly the trainable weights (252 A's, 252 B's, the head's 2 tensors)
    trainable = {name for name, p in teacher.named_parameters() if p.requires_grad}   ## {...} with "for": a set
    assert not result.unexpected_keys, f"Names in the file that the model doesn't have: {result.unexpected_keys[:3]}"
    assert set(saved) == trainable, "The file doesn't hold exactly the model's trainable weights"
    print(f"Loaded {len(saved)} trained tensors ({sum(w.numel() for w in saved.values()) / 1e6:.1f}M numbers); the other "
          f"{len(result.missing_keys)} tensors are pre-trained Qwen's  ({(time.perf_counter() - start) / 60:.1f} min)")

    ## Dev first: check the teacher before spending 7 more minutes on the other posts
    dev = teacher_logits(teacher, tokenize(X_dev), qwen_tokenizer.pad_token_id)   ## (1109, 20)
    dev_acc = (dev.argmax(axis=1) == y_dev).mean() * 100
    print(f"dev posts: accuracy {dev_acc:.2f} (the LoRA notebook: 82.24)  ({(time.perf_counter() - start) / 60:.1f} min)")
    assert abs(dev_acc - 82.24) < 0.5, "The rebuilt teacher doesn't match the LoRA notebook's dev accuracy"

    train = teacher_logits(teacher, tokenize(X_tr), qwen_tokenizer.pad_token_id)                  ## (9987, 20)
    print(f"training posts done  ({(time.perf_counter() - start) / 60:.1f} min)")
    test = teacher_logits(teacher, tokenize(newsgroups_test.data), qwen_tokenizer.pad_token_id)   ## (7370, 20)
    print(f"test posts done  ({(time.perf_counter() - start) / 60:.1f} min). "
          f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    return train, dev, test

## Part 1b: an honest teacher (frozen Qwen + LR, out of fold)

Run 1–5 used the LoRA teacher, and distillation closed none of the gap (see Result). One reason: that teacher was trained on the very posts the student imitates it on, for 3 epochs, and memorized them. On those posts it's 98.2% right, and its median probability for all the wrong classes together is 0.006%: its soft labels are the hard labels. Cho & Hariharan 2019 found that "more accurate teachers often don't make good teachers", and that stopping the teacher's training early helps.

This teacher never saw the posts it labels:
- **The frozen Qwen3-Embedding-4B vectors, already cached** (`../frozen/qwen3_4b_2048_instrB_last_20ng.npz`: 2,048 tokens, the same instruction), standardized, with the logistic regression setting tuned on thread-grouped folds (C = 0.001, `../frozen/hparam_search_mean/lr_search_threads.json`).
- **Out of fold:** each training post's soft label comes from an LR fitted on the other 9 of 10 thread-grouped folds. So it says how frozen Qwen sees a post *it hasn't trained on*: 79.3% right, with a median 10.8% of the probability off the true class.
- **Dev and test:** an LR fitted on all 9,987 training posts (dev 81.79, test 77.72; the LoRA teacher: 82.24 and 78.68).
- **Calibrated:** its confidence matches its accuracy to within 1–3 points, while the LoRA teacher is 11–13 points overconfident. So $T = 1$ uses its probabilities as they are.
- **"Logits" = log-probabilities.** The softmax ignores a constant added to every logit of a post, so $\text{softmax}(\log p) = p$, and at temperature $T$ it gives the same distribution as the original logits would.

About a minute on the CPU; set `TEACHER = "frozen_lr_oof"` to use it.

In [5]:
from threadpoolctl import threadpool_limits
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

## Frozen Qwen + LR's log-probabilities (used as logits) for the training, dev and test posts. out_of_fold=False is a
## control (Part 1c): the training posts are labeled by the LR fitted on them.
def compute_lr_teacher_logits(out_of_fold=True):
    vectors = np.load("../frozen/qwen3_4b_2048_instrB_last_20ng.npz")   ## frozen Qwen: (11096, 2560) training, (7370, 2560) test
    X_all, X_te = vectors["X_train"], vectors["X_test"]
    X_tr_vec = X_all[train_idx]                                    ## (9987, 2560), in the same order as X_tr

    def fit(X, y):   ## standardize, then LR with the tuned C. One BLAS thread: ~60x faster on this machine (Lessons.md)
        scaler = StandardScaler().fit(X)
        with threadpool_limits(1):
            return scaler, LogisticRegression(C=0.001, max_iter=2000).fit(scaler.transform(X), y)

    def log_probs(model, X):   ## (N, 2560) vectors -> (N, 20) log-probabilities; clip so that log never sees 0
        scaler, lr = model
        return np.log(np.clip(lr.predict_proba(scaler.transform(X)), 1e-12, None)).astype(np.float32)

    full = fit(X_tr_vec, y_tr)   ## all 9,987 training posts, for the dev and test posts
    if not out_of_fold:          ## the control: the same LR labels the posts it was fitted on
        return log_probs(full, X_tr_vec), log_probs(full, X_all[dev_idx]), log_probs(full, X_te)

    ## Out of fold: 10 folds that never split a thread (like the dev split); each fold's posts are labeled by an LR
    ## fitted on the other 9 folds
    train = np.zeros((len(train_idx), n_classes), dtype=np.float32)
    folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)
    for fit_rows, held_out in folds.split(X_tr_vec, y_tr, groups=thread_ids[train_idx]):
        train[held_out] = log_probs(fit(X_tr_vec[fit_rows], y_tr[fit_rows]), X_tr_vec[held_out])
    return train, log_probs(full, X_all[dev_idx]), log_probs(full, X_te)

## Part 1c: two controls for the +1.64

The frozen Qwen + LR teacher (Part 1b) raised the student from 73.13 to 74.77 on test. Each control repeats that run exactly (α = 0.5, T = 1, seeds 423, 1 and 2) with one change, to find out *why* it helped:
- **`"label_smoothing"`: a teacher that knows nothing.**
  - Every training post gets the same flat soft label: 1 − ε on its true class, plus ε spread evenly over all 20 classes (label smoothing, Szegedy et al. 2016).
  - ε is set so that these labels are exactly as soft *on average* as the frozen Qwen + LR teacher's. They just say nothing about which classes a post resembles, or which posts are ambiguous.
  - If this gains as much, the +1.64 was regularization from softer targets (Yuan et al. 2020), not Qwen's knowledge.
- **`"frozen_lr_insample"`: the same teacher, but labeling the posts it was fitted on,** instead of out of fold.
  - In-sample it's 88.0% right with a median 8.7% of the probability off the true class, against 79.3% and 10.8% out of fold. It's strongly regularized (C = 0.001), so it memorizes far less than LoRA Qwen (98.2%, 0.006%).
  - If this gains as much, out-of-fold labels weren't needed for this teacher. If much less, even mild memorization costs a lot.

For both, the dev and test "teacher logits" are frozen Qwen + LR's. They're never used in training, only to measure agreement with that teacher. So "agreement" means the same thing in every run of this teacher family.

In [6]:
## Control: label smoothing as a "teacher" that knows nothing about the posts. Each training post's soft label is
## (1 - eps) on its true class plus eps / 20 on every class, with eps chosen to match the frozen_lr_oof teacher's
## average softness. Returns log-probabilities, like the other teachers. Needs the frozen_lr_oof logits file.
def label_smoothing_logits():
    oof = np.load(TEACHER_FILES["frozen_lr_oof"])
    p_oof = np.exp(oof["train"])                                  ## (9987, 20): that teacher's probabilities
    off = 1 - p_oof[np.arange(len(y_tr)), y_tr].mean()           ## its average probability off the true class
    eps = off * n_classes / (n_classes - 1)                       ## label smoothing puts eps * 19/20 off the true class
    smooth = np.full((len(y_tr), n_classes), eps / n_classes)     ## (9987, 20), every entry eps / 20
    smooth[np.arange(len(y_tr)), y_tr] += 1 - eps                 ## + (1 - eps) on each post's true class
    print(f"label smoothing: eps = {eps:.3f}, so {off * 100:.1f}% of every label's probability is off the true class")
    return np.log(smooth).astype(np.float32), oof["dev"], oof["test"]

## Part 1d: a stronger frozen teacher, frozen Qwen + MLP (out of fold)

Is a stronger teacher a better teacher, with everything else kept the same? Frozen Qwen + LR (Part 1b) scores about 78.0 on test. The tuned MLP on the same frozen vectors scores 78.80 ± 0.22 (`../frozen/hparam_search_mean/Qwen4B2048Search.md`). This part swaps the LR for that MLP, and keeps the rest of Part 1b: the same vectors, the same 10 thread-grouped folds and out-of-fold soft labels, and T = 1, α = 0.5.

**The MLP is the tuned one, trained exactly as the frozen-feature search trained it:**
- **Training** (`../frozen/hparam_search/search.py`, `train_model`): standardized inputs, input dropout 0.3, then 2,560 → 4,096 → 64 → 20 with GELU and dropout 0.2 and 0.7. AdamW at learning rate 1e-5 with weight decay 0.1, in batches of 32.
- **Early stopping:** on the cross-entropy of 10% of the threads, with patience 10, keeping the best epoch's weights.
- **Config:** `../frozen/hparam_search_mean/final_test_qwen3_4b_2048_instrB_threads.json`.

**One difference that matters for distillation: the tuned MLP was trained with label smoothing 0.3.**
- Its probabilities are softened by design, so this isn't purely "weaker vs. stronger": the two teachers' soft labels also differ in shape.
- Müller et al. 2019 found that a teacher trained with label smoothing distills much worse, because smoothing erases information from its logits. So if this teacher does worse than the LR despite being more accurate, that's the first suspect.

11 MLP fits on the GPU: one per fold, plus one on all 9,987 posts for dev and test. About 10 minutes.

**Measured: it's stronger but far too unsure of itself.**
- Out of fold it's 79.85% right (LR 79.28), with dev 82.15 (81.79) and test 78.22 (77.72).
- But its median top-class probability is only 0.56, and its confidence is about 30 points *below* its accuracy (the LR's is within 1–3). As soft labels, it puts 54% of the probability off the true class, against the LR's 29%.
- So `"frozen_mlp_oof_calibrated"` first applies **temperature scaling** (Guo et al. 2017): it divides the MLP's logits by the single temperature that minimizes its dev loss, 0.53, which sharpens them. Then the MLP is as calibrated as the LR (2–3 points), and about as soft (27% of the probability off the true class).
- That makes LR vs. calibrated MLP close to a pure "weaker vs. stronger teacher" comparison. The temperature is fitted on dev (as Guo et al. do), never on test.

In [7]:
MLP_CFG = {"dropout_in": 0.3, "hidden1": 4096, "dropout1": 0.2, "hidden2": 64, "dropout2": 0.7,
           "learning_rate": 1e-5, "weight_decay": 0.1, "batch_size": 32, "label_smoothing": 0.3}   ## the tuned config

## Trains one MLP the way the frozen-feature search did, and returns a function that maps (N, 2560) vectors to (N, 20)
## log-probabilities. groups: each post's thread, so that the early-stopping posts are 10% of the threads.
def fit_mlp(X, y, groups, seed):
    fit_rows, stop_rows = next(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423).split(X, y, groups))
    scaler = StandardScaler().fit(X[fit_rows])        ## standardize with the fitting posts' means and stds
    def to_gpu(vectors):                               ## standardized (N, 2560) float32 tensor in GPU memory
        return torch.tensor(scaler.transform(vectors), dtype=torch.float32, device=device)
    X_fit, X_stop = to_gpu(X[fit_rows]), to_gpu(X[stop_rows])
    y_fit, y_stop = torch.tensor(y[fit_rows], device=device), torch.tensor(y[stop_rows], device=device)

    torch.manual_seed(seed)
    c = MLP_CFG
    mlp = nn.Sequential(nn.Dropout(c["dropout_in"]),
                        nn.Linear(2560, c["hidden1"]), nn.GELU(), nn.Dropout(c["dropout1"]),
                        nn.Linear(c["hidden1"], c["hidden2"]), nn.GELU(), nn.Dropout(c["dropout2"]),
                        nn.Linear(c["hidden2"], n_classes)).to(device)
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=c["learning_rate"], weight_decay=c["weight_decay"])
    train_loss = nn.CrossEntropyLoss(label_smoothing=c["label_smoothing"])
    best_loss, best_epoch = float("inf"), 0
    for epoch in range(1, 301):
        mlp.train()
        order = torch.randperm(len(X_fit), device=device)
        for i in range(0, len(order), c["batch_size"]):
            idx = order[i:i + c["batch_size"]]
            optimizer.zero_grad()
            train_loss(mlp(X_fit[idx]), y_fit[idx]).backward()
            optimizer.step()
        mlp.eval()
        with torch.no_grad():   ## early stopping on plain cross-entropy, as search.py does
            stop_loss = nn.functional.cross_entropy(mlp(X_stop), y_stop).item()
        if stop_loss < best_loss:
            best_loss, best_epoch = stop_loss, epoch
            best_state = {k: v.clone() for k, v in mlp.state_dict().items()}
        elif epoch - best_epoch >= 10:   ## 10 epochs without a new lowest loss: stop
            break
    mlp.load_state_dict(best_state)
    mlp.eval()

    def log_probs(vectors):
        with torch.no_grad():
            return torch.log_softmax(mlp(to_gpu(vectors)), dim=1).cpu().numpy()
    return log_probs

## Frozen Qwen + MLP's out-of-fold log-probabilities for the training posts, and the full fit's for dev and test
def compute_mlp_teacher_logits():
    vectors = np.load("../frozen/qwen3_4b_2048_instrB_last_20ng.npz")
    X_all, X_te = vectors["X_train"], vectors["X_test"]
    X_tr_vec, groups = X_all[train_idx], thread_ids[train_idx]   ## (9987, 2560) and each post's thread
    start = time.perf_counter()
    train = np.zeros((len(train_idx), n_classes), dtype=np.float32)
    folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)   ## the same folds as the LR teacher
    for k, (fit_rows, held_out) in enumerate(folds.split(X_tr_vec, y_tr, groups=groups)):
        predict = fit_mlp(X_tr_vec[fit_rows], y_tr[fit_rows], groups[fit_rows], seed=10_000 + k)
        train[held_out] = predict(X_tr_vec[held_out])
        print(f"  fold {k + 1}/10 done  ({(time.perf_counter() - start) / 60:.1f} min)")
    predict = fit_mlp(X_tr_vec, y_tr, groups, seed=10_010)   ## all 9,987 training posts, for dev and test
    return train, predict(X_all[dev_idx]), predict(X_te)

## The same MLP teacher after temperature scaling (Guo et al. 2017): divide its logits by the one temperature that
## minimizes its dev loss (searched from 0.10 to 1.50). Needs the frozen_mlp_oof logits file.
def calibrated_mlp_logits():
    mlp = np.load(TEACHER_FILES["frozen_mlp_oof"])
    def dev_loss(t):   ## cross-entropy of softmax(dev logits / t) with the true labels
        return nn.functional.cross_entropy(torch.from_numpy(mlp["dev"]) / t, torch.from_numpy(y_dev)).item()
    t = min(np.arange(0.10, 1.51, 0.01), key=dev_loss)
    print(f"temperature scaling: dividing the MLP's logits by {t:.2f} (dev loss {dev_loss(1.0):.3f} -> {dev_loss(t):.3f})")
    t = np.float32(t)   ## keep the logits float32 (a float64 t would turn them into float64)
    return mlp["train"] / t, mlp["dev"] / t, mlp["test"] / t

In [8]:
import gc

if TEACHER_FILE.exists():
    cached = np.load(TEACHER_FILE)   ## the arrays that np.savez stored, by name, like a dict
    ## The saved logits are only valid for this split: check that they're for the same posts, in the same order
    assert np.array_equal(cached["train_idx"], train_idx) and np.array_equal(cached["dev_idx"], dev_idx)
    teacher_tr, teacher_dev, teacher_test = cached["train"], cached["dev"], cached["test"]
    print(f"Loaded the teacher's logits from {TEACHER_FILE}")
else:
    if TEACHER == "lora":
        teacher_tr, teacher_dev, teacher_test = compute_teacher_logits()
    elif TEACHER == "frozen_lr_oof":
        teacher_tr, teacher_dev, teacher_test = compute_lr_teacher_logits(out_of_fold=True)
    elif TEACHER == "frozen_lr_insample":
        teacher_tr, teacher_dev, teacher_test = compute_lr_teacher_logits(out_of_fold=False)
    elif TEACHER == "frozen_mlp_oof":
        teacher_tr, teacher_dev, teacher_test = compute_mlp_teacher_logits()
    elif TEACHER == "frozen_mlp_oof_calibrated":
        teacher_tr, teacher_dev, teacher_test = calibrated_mlp_logits()
    else:   ## "label_smoothing"
        teacher_tr, teacher_dev, teacher_test = label_smoothing_logits()
    ## Nothing refers to the teacher now. gc.collect() makes Python free it right away, and empty_cache() hands back
    ## the GPU memory that PyTorch kept cached for reuse (the LoRA notebook's Evaluation section explains that cache).
    gc.collect()
    torch.cuda.empty_cache()
    ## np.savez stores several arrays in one file, each under a name. train_idx and dev_idx record the split.
    np.savez(TEACHER_FILE, train=teacher_tr, dev=teacher_dev, test=teacher_test, train_idx=train_idx, dev_idx=dev_idx)
    print(f"Saved the teacher's logits to {TEACHER_FILE}. GPU memory still in use: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("teacher logits: training", teacher_tr.shape, " dev", teacher_dev.shape, " test", teacher_test.shape)

Loaded the teacher's logits from qwen3-embedding-4b-frozen-mlp-oof-calibrated-logits.npz
teacher logits: training (9987, 20)  dev (1109, 20)  test (7370, 20)


## What the teacher's soft labels look like

Two predictions before running the next cell:
- At $T = 1$, the teacher's average probability for its top class, on training posts and on dev posts. Which is higher, and by how much? (Its training loss ended at 0.146, its dev loss at 0.897.)
- How much do $T = 2$ and $T = 4$ flatten that?

In [9]:
## The teacher's probabilities at temperature T: softmax(z / T) along each row, (N, 20) logits -> (N, 20) probabilities
def soft_labels(logits, T):
    return torch.softmax(torch.from_numpy(logits) / T, dim=1).numpy()

print(f"Teacher accuracy: training posts {(teacher_tr.argmax(axis=1) == y_tr).mean() * 100:.2f}, "
      f"dev posts {(teacher_dev.argmax(axis=1) == y_dev).mean() * 100:.2f}\n")
print("Average probability of the teacher's top class:")
print(f"{'':16s}{'T = 1':>8s}{'T = 2':>8s}{'T = 4':>8s}")   ## {'':16s}: 16 spaces; {'T = 1':>8s}: right-aligned in 8
for name, z in [("training posts", teacher_tr), ("dev posts", teacher_dev)]:
    ## soft_labels(z, T).max(axis=1): each post's largest probability, (N,); .mean() averages them
    print(f"{name:16s}" + "".join(f"{soft_labels(z, T).max(axis=1).mean():8.3f}" for T in (1, 2, 4)))

Teacher accuracy: training posts 79.85, dev posts 82.15

Average probability of the teacher's top class:
                   T = 1   T = 2   T = 4
training posts     0.814   0.467   0.185
dev posts          0.833   0.480   0.188


In [10]:
## One training post: its first 300 characters, then the teacher's top 5 classes at T = 1, 2 and 4
i = 0
print(f"Training post {i}, true class: {names[y_tr[i]]}")
## .split() cuts the text at every run of spaces and newlines; " ".join glues the pieces back with single spaces
print(" ".join(X_tr[i].split())[:300], "...\n")
for T in (1, 2, 4):
    p = soft_labels(teacher_tr[i:i + 1], T)[0]   ## [i:i + 1] keeps row i as a (1, 20) array; [0] takes it out: (20,)
    top5 = np.argsort(p)[::-1][:5]               ## argsort: smallest to largest; [::-1] reverses; [:5]: the 5 largest
    print(f"T = {T}:  " + ",  ".join(f"{names[c]} {p[c]:.3f}" for c in top5))

Training post 0, true class: rec.autos
I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ early 70s. It was called a Bricklin. The doors were really small. In addition, the front bumper was separate from the rest of the body. This is all I kn ...

T = 1:  rec.autos 0.976,  comp.sys.mac.hardware 0.003,  talk.politics.guns 0.002,  sci.electronics 0.002,  rec.motorcycles 0.002
T = 2:  rec.autos 0.598,  comp.sys.mac.hardware 0.031,  talk.politics.guns 0.028,  sci.electronics 0.026,  rec.motorcycles 0.026
T = 4:  rec.autos 0.219,  comp.sys.mac.hardware 0.050,  talk.politics.guns 0.048,  sci.electronics 0.045,  rec.motorcycles 0.045


## Part 2: the student

`Fine-tuned Roberta.ipynb` from here on, with its comments shortened, apart from three changes:
1. **Each training batch also carries the teacher's logits** for its posts: a 4th tensor from the DataLoader.
2. **The loss** is `distillation_loss`, defined below, instead of cross-entropy alone.
3. **Each epoch also measures agreement with the teacher:** the share of dev posts where the student picks the teacher's class.

**Shapes:**
- **N** = posts in a batch (16 in training, 64 in evaluation)
- **L** = tokens in the batch's longest post, at most 512 (the other notebooks called it T, but here $T$ is the temperature)
- **768** = the size of RoBERTa's vectors; **20** = classes

In [11]:
from torch.utils.data import DataLoader

torch.manual_seed(SEED)   ## SEED sets the batch order, the head's starting weights and the dropout
MAX_LENGTH = 512
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

## A list of N (text, label, teacher logits) triples -> input_ids (N, L), attention_mask (N, L), labels (N,) and the
## teacher's logits (N, 20): the full fine-tune's collate, plus the 4th tensor
def collate(batch):
    texts, labels, t_logits = zip(*batch)   ## three tuples of N: the texts, the labels, and N rows of 20 logits
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    ## np.stack piles the N (20,) rows into one (N, 20) array; torch.from_numpy makes it a tensor
    return enc["input_ids"], enc["attention_mask"], torch.tensor(labels), torch.from_numpy(np.stack(t_logits))

train_lengths = torch.tensor([len(ids) for ids in tokenizer(X_tr, truncation=True, max_length=MAX_LENGTH)["input_ids"]])

## Batches of posts of similar length, in a new random order every epoch (the full fine-tune explains how)
class LengthGroupedBatches:
    def __init__(self, lengths, batch_size):
        self.lengths, self.batch_size = lengths, batch_size

    def __len__(self):
        return -(-len(self.lengths) // self.batch_size)

    def __iter__(self):
        order = torch.randperm(len(self.lengths))
        batches = []
        for mega in order.split(50 * self.batch_size):
            mega = mega[self.lengths[mega].argsort(descending=True)]
            batches += mega.split(self.batch_size)
        for i in torch.randperm(len(batches)):
            yield batches[i].tolist()

## zip(X_tr, y_tr, teacher_tr) pairs each post with its label and its row of teacher logits: (text, label, (20,) array).
## Dev and test carry the teacher's logits too, so that one collate serves all three loaders.
train_loader = DataLoader(list(zip(X_tr, y_tr, teacher_tr)), batch_sampler=LengthGroupedBatches(train_lengths, 16),
                          collate_fn=collate)                                                                    ## 625 batches
dev_loader = DataLoader(list(zip(X_dev, y_dev, teacher_dev)), batch_size=64, collate_fn=collate)                 ## 18 batches
test_loader = DataLoader(list(zip(newsgroups_test.data, y_test, teacher_test)), batch_size=64, collate_fn=collate)  ## 116

input_ids, attention_mask, labels, t_logits = next(iter(train_loader))   ## one training batch, for the checks below
print("token ids:", tuple(input_ids.shape), " attention mask:", tuple(attention_mask.shape), " labels:", tuple(labels.shape),
      " teacher logits:", tuple(t_logits.shape))

token ids: (16, 31)  attention mask: (16, 31)  labels: (16,)  teacher logits: (16, 20)


In [12]:
## The full fine-tune's model, unchanged: roberta-base without its pooler, then Hugging Face's head on <s>
class RobertaClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.roberta = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False)
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(768, 768), nn.Tanh(),     ## new and random: (N, 768) -> (N, 768)
            nn.Dropout(0.1),
            nn.Linear(768, n_classes))          ## new and random: (N, 768) -> (N, 20)

    def forward(self, input_ids, attention_mask):    ## both (N, L)
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state   ## (N, L, 768)
        return self.head(self.dropout(hidden[:, 0]))  ## <s>: (N, 768) -> (N, 20)

model = RobertaClassifier(n_classes).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters, all of them trainable")

124.7M parameters, all of them trainable


## The distillation loss

The formula from the top, in code. Three PyTorch details:
- **`F.log_softmax(z / T)`** computes the log of the softmax directly, in log-space (Lecture 4). It never exponentiates a large logit into a huge number, or rounds a small probability down to 0.
- **`F.kl_div(input, target)` takes its arguments in the opposite order from the math.** $\text{KL}(\text{teacher} \,\|\, \text{student})$ is written teacher first, but `F.kl_div` wants the student's log-probabilities first and the teacher's second. Swapped, it silently computes something else, $\text{KL}(\text{student} \,\|\, \text{teacher})$.
- **`reduction="batchmean"`:** sum over the 20 classes, then average over the N posts, which gives the KL divergence per post. The default, `"mean"`, divides by N × 20 instead, which isn't a KL divergence at all (PyTorch's documentation warns about this).

Then three checks on the demo batch, before any training. Predict first:
- The untrained student's hard loss. (Its head is random, so its 20 probabilities are all close to 1/20.)
- The soft loss of a student that matched the teacher exactly.

In [13]:
import torch.nn.functional as F

## Returns (total loss, hard part, soft part), each a 0-dim tensor (one number). student_logits and teacher_logits:
## (N, 20) float32; labels: (N,). T and alpha default to this run's settings; the checks below pass others.
def distillation_loss(student_logits, teacher_logits, labels, T=TEMPERATURE, alpha=ALPHA):
    ## The hard part: the plain fine-tune's loss, cross-entropy with the true labels (at T = 1)
    hard = F.cross_entropy(student_logits, labels)
    ## The soft part: both distributions at temperature T, as log-probabilities, (N, 20) each
    log_p_student = F.log_softmax(student_logits / T, dim=1)
    log_p_teacher = F.log_softmax(teacher_logits / T, dim=1)
    ## KL(teacher || student), averaged over the N posts: the student's log-probabilities go first (see above).
    ## log_target=True: the target is given as log-probabilities too. Times T², as in Hinton's Section 2.
    soft = F.kl_div(log_p_student, log_p_teacher, reduction="batchmean", log_target=True) * T**2
    return alpha * hard + (1 - alpha) * soft, hard, soft

## The demo batch from above, through the untrained student: eval mode (no dropout), no gradients
ids, mask = input_ids.to(device), attention_mask.to(device)
y_b, t_b = labels.to(device), t_logits.to(device)       ## (N,), (N, 20)
model.eval()
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    s_b = model(ids, mask).float()                        ## (N, 20): the untrained student's logits
total, hard, soft = distillation_loss(s_b, t_b, y_b)
print(f"untrained student: hard {hard.item():.3f}, soft {soft.item():.3f}, "
      f"total = {ALPHA} x hard + {1 - ALPHA} x soft = {total.item():.3f}")

## Check 1: a student identical to the teacher. The KL divergence of a distribution with itself is 0.
## distillation_loss(...)[2] is the third value it returns: the soft part.
print("student = teacher -> soft part:", distillation_loss(t_b, t_b, y_b)[2].item())
## Check 2: alpha = 1 leaves only the hard part, the plain fine-tune's loss
print("alpha = 1 gives plain cross-entropy:",
      torch.allclose(distillation_loss(s_b, t_b, y_b, alpha=1.0)[0], F.cross_entropy(s_b, y_b)))
## Check 3: F.kl_div against the definition, the sum over c of p_c log(p_c / q_c) (J&M eq. 22.38), averaged over posts
p_t = F.softmax(t_b / TEMPERATURE, dim=1)                                                        ## (N, 20)
log_ratio = F.log_softmax(t_b / TEMPERATURE, dim=1) - F.log_softmax(s_b / TEMPERATURE, dim=1)   ## log(p_c / q_c)
by_hand = (p_t * log_ratio).sum(dim=1).mean() * TEMPERATURE**2
print("F.kl_div matches the formula:", torch.allclose(by_hand, soft, rtol=1e-4))

untrained student: hard 2.990, soft 1.434, total = 0.5 x hard + 0.5 x soft = 2.212
student = teacher -> soft part: 0.0
alpha = 1 gives plain cross-entropy: True
F.kl_div matches the formula: True


## Optimizer and learning-rate schedule

The RoBERTa paper's recipe, unchanged from the full fine-tune: AdamW at learning rate 2e-5 with weight decay 0.1, β₂ 0.98 and ε 1e-6, warmup over the first 6% of the steps, then linear decay to 0, for 10 epochs.

In [14]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 10
LR = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.98), eps=1e-6)
total_steps = EPOCHS * len(train_loader)   ## 10 x 625 = 6250
warmup_steps = int(0.06 * total_steps)     ## 375
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()          ## only for the dev loss: cross-entropy with the true labels, as in the other runs
print(f"{len(train_loader)} batches per epoch, {total_steps} steps in total, {warmup_steps} of them warmup")

625 batches per epoch, 6250 steps in total, 375 of them warmup


## Evaluation

As in `LoRA Roberta.ipynb`: dropout off, no gradients, bfloat16, and it also returns every post's 20 probabilities. It ignores the teacher's logits that the DataLoader hands it. The dev loss is cross-entropy with the true labels, so it can be compared with the other runs' dev loss.

In [15]:
## Returns (mean loss, accuracy, probabilities (number of posts, 20)) over all the batches of one DataLoader
def evaluate(loader):
    model.eval()
    logits, labels = [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for input_ids, attention_mask, y, _ in loader:   ## _ = the teacher's logits, not needed here
            logits.append(model(input_ids.to(device), attention_mask.to(device)).float().cpu())   ## (N, 20)
            labels.append(y)
    logits, labels = torch.cat(logits), torch.cat(labels)   ## (number of posts, 20), (number of posts,)
    probs = logits.softmax(dim=1)
    return criterion(logits, labels).item(), (probs.argmax(dim=1) == labels).float().mean().item(), probs.numpy()

## Training loop

The full fine-tune's loop, with the loss swapped for `distillation_loss`. It prints both parts of the loss, so we can watch the student fit the labels (hard) and the teacher (soft). After each epoch it prints the dev scores and the **agreement**: the share of dev posts where the student picks the same class as the teacher. The first line gives the plain fine-tune's agreement, for comparison.

As in the plain fine-tune runs we compare with, the epoch kept is the one with the best dev accuracy.

Time: about 8 minutes (measured: 48 s per epoch).

In [16]:
import time

## For comparison: the plain fine-tune's dev scores (sweep.py's seed-423 run, at its best epoch)
teacher_dev_pred = teacher_dev.argmax(axis=1)                                   ## (1109,): the teacher's class per post
plain_dev_pred = np.load("../sweeps/probs/r_lr2e-5_seed423_acc_dev.npy").argmax(axis=1)   ## (1109,)
print(f"Plain fine-tune: dev acc {(plain_dev_pred == y_dev).mean() * 100:.2f}, "
      f"agreement with the teacher {(plain_dev_pred == teacher_dev_pred).mean() * 100:.2f}\n")

best = {"dev_acc": 0.0, "epoch": 0, "state": None}
history = []   ## one dict per epoch, in sweep.py's format, plus the loss parts and the agreement
torch.cuda.reset_peak_memory_stats()   ## so that the peak printed at the end is the student's, not the teacher's
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    sums = {"total": 0.0, "hard": 0.0, "soft": 0.0}   ## each part of the loss, summed over the epoch's posts
    for step, (input_ids, attention_mask, labels, t_logits) in enumerate(train_loader, start=1):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)   ## (N, L) each
        labels, t_logits = labels.to(device), t_logits.to(device)                     ## (N,), (N, 20)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(input_ids, attention_mask)                                 ## (N, 20)
        ## The one change from the full fine-tune: the student learns from the labels AND the teacher's logits. Those
        ## are plain numbers loaded from disk, so the gradient flows only into the student.
        loss, hard, soft = distillation_loss(logits.float(), t_logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        for part, value in (("total", loss), ("hard", hard), ("soft", soft)):
            sums[part] += value.item() * len(labels)
        if step % 100 == 0:
            print(f"  epoch {epoch}, step {step}/{len(train_loader)}: loss {loss.item():.3f} "
                  f"(hard {hard.item():.3f}, soft {soft.item():.3f}), learning rate {scheduler.get_last_lr()[0]:.1e}")

    dev_loss, dev_acc, dev_probs = evaluate(dev_loader)
    dev_agree = float((dev_probs.argmax(axis=1) == teacher_dev_pred).mean())   ## share of dev posts given the teacher's class
    n = len(X_tr)
    history.append({"epoch": epoch, "train_loss": round(sums["total"] / n, 4), "train_hard": round(sums["hard"] / n, 4),
                    "train_soft": round(sums["soft"] / n, 4), "dev_loss": round(dev_loss, 4),
                    "dev_acc": round(dev_acc * 100, 2), "dev_agree": round(dev_agree * 100, 2),
                    "seconds": round(time.perf_counter() - start)})
    print(f"epoch {epoch}: train loss {sums['total'] / n:.3f} (hard {sums['hard'] / n:.3f}, soft {sums['soft'] / n:.3f})  "
          f"dev loss {dev_loss:.3f}  dev acc {dev_acc * 100:.2f}  agreement {dev_agree * 100:.2f}  "
          f"({time.perf_counter() - start:.0f} s so far)")
    if dev_acc >= best["dev_acc"]:   ## >=: on a tie, keep the later epoch
        best = {"dev_acc": dev_acc, "epoch": epoch,
                "state": {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}}

model.load_state_dict(best["state"])
print(f"Using epoch {best['epoch']} (highest dev accuracy). Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Plain fine-tune: dev acc 78.00, agreement with the teacher 84.49



  epoch 1, step 100/625: loss 2.877 (hard 3.059, soft 2.695), learning rate 5.3e-06


  epoch 1, step 200/625: loss 1.967 (hard 2.392, soft 1.542), learning rate 1.1e-05


  epoch 1, step 300/625: loss 1.153 (hard 1.382, soft 0.924), learning rate 1.6e-05


  epoch 1, step 400/625: loss 0.778 (hard 0.883, soft 0.672), learning rate 2.0e-05


  epoch 1, step 500/625: loss 0.756 (hard 1.222, soft 0.289), learning rate 2.0e-05


... (61 outputs left out)

  epoch 10, step 400/625: loss 0.201 (hard 0.241, soft 0.161), learning rate 7.7e-07


  epoch 10, step 500/625: loss 0.100 (hard 0.131, soft 0.069), learning rate 4.3e-07


  epoch 10, step 600/625: loss 0.288 (hard 0.280, soft 0.297), learning rate 8.5e-08


epoch 10: train loss 0.221 (hard 0.254, soft 0.189)  dev loss 0.756  dev acc 78.99  agreement 86.29  (469 s so far)
Using epoch 9 (highest dev accuracy). Peak GPU memory: 5.2 GB


## Save the probabilities; test only in the final runs

The best epoch's dev and test probabilities go to `../sweeps/probs/`, named the way `sweep.py` names its files, and the run's dev history goes to `../sweeps/sweep.jsonl`, so `summarize.py` lists it next to the other runs. The test accuracy prints only with `SCORE_TEST = True`: while choosing the temperature, don't look at test.

In [17]:
import json
from sklearn import metrics

_, _, dev_probs = evaluate(dev_loader)                    ## (1109, 20)
test_loss, test_acc, test_probs = evaluate(test_loader)   ## (7370, 20)
Path("../sweeps/probs").mkdir(exist_ok=True)
np.save(f"../sweeps/probs/{NAME}_seed{SEED}_acc_dev.npy", dev_probs)
np.save(f"../sweeps/probs/{NAME}_seed{SEED}_acc_test.npy", test_probs)
record = {"name": NAME,
          "cfg": {"teacher": TEACHER, "temperature": TEMPERATURE, "alpha": ALPHA, "lr": LR,
                  "epochs": EPOCHS, "wd": 0.1, "warmup": 0.06, "beta2": 0.98, "eps": 1e-6, "seed": SEED, "select": "acc"},
          "history": history, "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1), "epoch_used": best["epoch"]}
with open("../sweeps/sweep.jsonl", "a") as f:   ## "a" appends a line, keeping the earlier runs
    f.write(json.dumps(record) + "\n")
print(f"Saved ../sweeps/probs/{NAME}_seed{SEED}_acc_{{dev,test}}.npy and added this run to ../sweeps/sweep.jsonl")

if SCORE_TEST:
    test_pred = test_probs.argmax(axis=1)
    print(f"\nDistilled RoBERTa test accuracy: {test_acc * 100:.2f}  "
          f"(macro-F1 {metrics.f1_score(y_test, test_pred, average='macro') * 100:.2f})")
    print(f"Agreement with the teacher on test: {(test_pred == teacher_test.argmax(axis=1)).mean() * 100:.2f}")
else:
    print("Test not scored (SCORE_TEST = False)")

Saved probs/kd_frozen_mlp_oof_calibrated_T1_a0.5_seed2_acc_{dev,test}.npy and added this run to sweep.jsonl

Distilled RoBERTa test accuracy: 74.33  (macro-F1 73.04)
Agreement with the teacher on test: 81.26


## Final comparison: after all 3 seeds

Once the chosen temperature has run with seeds 423, 1 and 2, this cell compares its test scores with the plain fine-tune's (`r_lr2e-5`, the same 3 seeds) and with the teacher, and applies the decision rule from "How to run it". Until then it only says what's missing. It reads test, so run it only after the temperature has been chosen on dev.
- **Fraction of the gap closed:** (distilled mean − plain mean) / (teacher − plain mean).
- **McNemar's test** (`Lessons.md`) looks only at the test posts that exactly one of the two ensembles gets right. If the two were equally good, each would win about half of those posts, like a fair coin; `binomtest` gives the probability of a split at least this uneven.
- **Agreement with the teacher on test,** for both students. If distillation raises it a lot but accuracy only a little, the student copied the teacher's mistakes along with its successes.

In [18]:
from scipy.stats import binomtest

SEEDS = (423, 1, 2)
files = [Path(f"../sweeps/probs/{NAME}_seed{s}_acc_test.npy") for s in SEEDS]
if not all(f.exists() for f in files):
    print(f"Not yet: {NAME} needs all 3 seeds {SEEDS}; found {sum(f.exists() for f in files)}.")
else:
    distilled = [np.load(f) for f in files]                                    ## 3 arrays of (7370, 20) probabilities
    plain = [np.load(f"../sweeps/probs/r_lr2e-5_seed{s}_acc_test.npy") for s in SEEDS]   ## the plain fine-tune's, same seeds
    teacher_pred = teacher_test.argmax(axis=1)                                  ## (7370,)

    def accuracy(probs):    ## (7370, 20) probabilities -> % of test posts right
        return (probs.argmax(axis=1) == y_test).mean() * 100

    def agreement(probs):   ## % of test posts where this model picks the teacher's class
        return (probs.argmax(axis=1) == teacher_pred).mean() * 100

    distilled_acc, plain_acc = [accuracy(q) for q in distilled], [accuracy(q) for q in plain]
    distilled_mean, plain_mean = np.mean(distilled_acc), np.mean(plain_acc)
    teacher_acc = (teacher_pred == y_test).mean() * 100
    gain = distilled_mean - plain_mean
    print(f"{'seed':>6s}{'plain':>9s}{'distilled':>11s}")
    for s, a, b in zip(SEEDS, plain_acc, distilled_acc):
        print(f"{s:>6d}{a:9.2f}{b:11.2f}")
    print(f"{'mean':>6s}{plain_mean:9.2f}{distilled_mean:11.2f}   gain {gain:+.2f}   teacher {teacher_acc:.2f}")
    print(f"Fraction of the gap closed: {gain:.2f} / {teacher_acc - plain_mean:.2f} = "
          f"{gain / (teacher_acc - plain_mean) * 100:.0f}%\n")

    ## Ensembles: average the 3 seeds' probabilities, then take each post's top class
    distilled_ens, plain_ens = np.mean(distilled, axis=0), np.mean(plain, axis=0)          ## (7370, 20) each
    d_right, p_right = distilled_ens.argmax(axis=1) == y_test, plain_ens.argmax(axis=1) == y_test   ## (7370,) True/False
    only_d, only_p = int((d_right & ~p_right).sum()), int((~d_right & p_right).sum())   ## & = and, ~ = not, elementwise
    p_value = binomtest(only_d, only_d + only_p).pvalue if only_d + only_p > 0 else 1.0
    print(f"Ensembles: plain {accuracy(plain_ens):.2f}, distilled {accuracy(distilled_ens):.2f}. Right only in the "
          f"distilled one: {only_d} posts, only in the plain one: {only_p}. McNemar p = {p_value:.3g}")
    print(f"Agreement with the teacher on test (mean of 3 seeds): plain {np.mean([agreement(q) for q in plain]):.2f}, "
          f"distilled {np.mean([agreement(q) for q in distilled]):.2f}\n")

    if gain < 1:
        print("Decision rule: under 1 point, no clear effect.")
    elif p_value < 0.05:
        print("Decision rule: 1 point or more, and p < 0.05: distillation helps.")
    else:
        print("Decision rule: 1 point or more, but p >= 0.05: inconclusive, more seeds needed.")

    print("\nFor context (fine-tuning-results.md; these read 2,048 tokens per post):")
    for name, acc in [("Qwen3-Embedding-4B, frozen + LR (tuned)", 78.00),
                      ("Qwen3-Embedding-4B, frozen + MLP (tuned, 10-seed ensemble)", 78.97)]:
        print(f"  {name:60s} {acc:6.2f}")

  seed    plain  distilled
   423    72.84      73.76
     1    73.11      74.87
     2    73.45      74.33
  mean    73.13      74.32   gain +1.19   teacher 78.22
Fraction of the gap closed: 1.19 / 5.09 = 23%

Ensembles: plain 73.93, distilled 75.03. Right only in the distilled one: 294 posts, only in the plain one: 213. McNemar p = 0.000371
Agreement with the teacher on test (mean of 3 seeds): plain 77.97, distilled 81.28

Decision rule: 1 point or more, and p < 0.05: distillation helps.

For context (fine-tuning-results.md; these read 2,048 tokens per post):
  Qwen3-Embedding-4B, frozen + LR (tuned)                       78.00
  Qwen3-Embedding-4B, frozen + MLP (tuned, 10-seed ensemble)    78.97


## Result

**Run 1: `kd_T2_a0.5`** (T = 2, α = 0.5, seed 423; dev only, test not read)

| epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| dev accuracy | 72.59 | 75.02 | 76.83 | 77.46 | 77.28 | 77.01 | 77.01 | 77.82 | 78.36 | **78.36** |
| agreement with the teacher | 75.02 | 79.17 | 80.79 | 81.15 | 81.51 | 81.97 | 80.97 | 82.51 | 82.69 | 82.06 |

- **Best dev accuracy 78.36 (epoch 10).** The plain fine-tune's 4 RoBERTa-recipe runs peaked at 78.00, 77.82 and 77.73 (`sweep.py`) and 77.55 (`Fine-tuned Roberta.ipynb`). So this run is 0.36 to 0.81 points above them, 4 to 9 of the 1,109 dev posts: a hint at best, from one run. The teacher scores 82.24.
- **The student didn't become more like the teacher.** At its best epoch it picked the teacher's class on 82.06% of the dev posts, against 83.50% for the plain fine-tune's seed-423 run.
- **On its training posts the teacher is almost always right, and sure of it:** 98.18% accurate, with an average top-class probability of 0.975 at T = 1, 0.924 at T = 2 and 0.658 at T = 4 (on dev posts: 0.928, 0.856 and 0.604). So at T = 2, the soft labels the student learned from left on average only 7.6% of the probability for the other 19 classes.
- **Cost:** the teacher's pass took 9.0 minutes (8.5 GB at most), and the student 8 minutes (48 s per epoch, 5.2 GB at most).

**Run 2: `kd_T4_a0.5`** (T = 4, α = 0.5, seed 423; dev only, test not read)

| epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| dev accuracy | 71.06 | 75.11 | 76.28 | 77.73 | 76.56 | 78.00 | 77.91 | 78.09 | **78.27** | 78.18 |
| agreement with the teacher | 74.39 | 79.35 | 80.43 | 81.79 | 80.97 | 82.69 | 83.23 | 83.23 | 83.32 | 83.14 |

- **Best dev accuracy 78.27 (epoch 9), one post behind T = 2's 78.36.** The two temperatures tie on dev, and both sit 0.3 to 0.8 points above the plain fine-tune's runs.
- **Agreement with the teacher rose a little,** to 83.32% at the kept epoch (T = 2: 82.06%), but only to the plain fine-tune's level (83.50%). Even with soft labels that leave 34% of the probability for the other classes, the student isn't more teacher-like on new posts than plain fine-tuning makes it.
- **Dev loss ran higher than at T = 2** (1.13 to 1.54, against 0.99 to 1.46), and the soft part of the training loss was still 0.91 after 10 epochs (T = 2: 0.27): at T = 4 the student is asked to match the whole shape of the teacher's 20 logits, and it doesn't get there.
- **Time:** 11.4 minutes, slower than run 1 because the GPU was shared with another job; peak memory 5.2 GB.

**Run 3: `kd_T1_a0.5`** (T = 1, α = 0.5, seed 423; dev only, test not read)

| epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| dev accuracy | 73.40 | 74.84 | 77.10 | **78.09** | 77.19 | 77.19 | 76.19 | 77.46 | 77.01 | 77.28 |
| agreement with the teacher | 76.56 | 79.62 | 81.15 | 81.51 | 81.33 | 80.79 | 81.15 | 81.97 | 81.88 | 81.70 |

- **Best dev accuracy 78.09, at epoch 4,** an early, noisy pick like the plain notebook run's epoch 3. After epoch 4 it behaved like the plain fine-tune: dev loss as low as 0.86 and 1.22 at the end (plain: 0.82 and 1.46), and a last epoch of 77.28, exactly the plain seed-423 run's. At T = 1 the soft labels are nearly the hard ones (the teacher's top class averages 0.975 on its training posts), so this is what T = 1 predicted.

**Choosing the temperature (step 3 of "How to run it")**

| | T = 1 | T = 2 | T = 4 | plain fine-tune (3 `sweep.py` seeds) |
|---|---|---|---|---|
| best-epoch dev accuracy | 78.09 | **78.36** | 78.27 | 78.00, 77.82, 77.73 |
| last-epoch dev accuracy | 77.28 | 78.36 | 78.18 | 77.28, 77.28, 77.46 |
| dev agreement with the teacher, kept epoch | 81.51 | 82.06 | 83.32 | 83.50 (seed 423) |

- **T = 2 is chosen:** the highest best-epoch dev accuracy, and the tie-break value anyway. The three temperatures span only 0.27 points (3 posts), so this choice hardly matters. T = 4 didn't win, so T = 8 isn't needed.
- **A hint, from one seed each:** at the last epoch, which avoids the noisy best-of-10 pick, T = 2 and T = 4 are about 1 point above the plain fine-tune's 3 runs and T = 1 (about 10 dev posts). The seeds 1 and 2 runs will show whether it holds.

**Seeds 1 and 2, then test** (T = 2; the comparison cell above ran in the seed-2 run)

| | seed 423 | seed 1 | seed 2 | mean | ensemble of 3 |
|---|---|---|---|---|---|
| best-epoch dev accuracy (epoch kept) | 78.36 (10) | 78.09 (10) | 77.82 (8) | 78.09 | |
| **test, distilled** | 72.94 | 73.58 | 72.93 | **73.15** (std 0.37) | **74.02** |
| test, plain fine-tune (`r_lr2e-5`) | 72.84 | 73.11 | 73.45 | 73.13 (std 0.31) | 73.93 |

**Conclusion: distillation from the training posts closed none of the gap.**
- **Gain +0.02 points on the 3-seed mean: 0% of the 5.55-point gap to the teacher (78.68).** The ensembles differ by 0.09: 218 test posts right only for the distilled one, 212 only for the plain one (McNemar p = 0.81). By the decision rule fixed before the runs: under 1 point, no clear effect. Macro-F1 moved the same way (72.03 vs. 71.87 on average).
- **Dev overstated it.** On dev, distillation was ahead by 0.24 points at the best epoch (mean of 3 seeds) and by about 0.7 at the last epoch (78.03 vs. 77.34). On test it was level. Dev comes from the training period, test from later.
- **The student barely became more like the teacher:** it picked the teacher's class on 78.05% of the test posts, against 77.34% for the plain fine-tune (on dev, 82–83% for both).
- **Why, most likely:** the student imitated the teacher only on the teacher's own training posts. There the teacher is 98.18% right and sure of it (top class 0.975 at T = 1, 0.924 at T = 2), so its soft labels say little that the hard labels don't, and the student learned nothing about how Qwen handles new posts. That matches the literature: in Patient KD (Sun et al. 2019, Table 1), vanilla distillation of BERT-base into a 6-layer BERT on the same labeled data closed between −6% and 43% of the gap on GLUE tasks (−0.2 points on MNLI). The larger shares came with augmented transfer sets (Tang et al. 2019; TinyBERT's ablation without augmentation lost 2 to 21 dev points) or with matching intermediate layers.
- **Not tested here:** the control for "Qwen's knowledge" vs. "soft labels regularize" isn't needed: there's no gain to explain.
- **Cost:** the teacher's pass once (9.0 minutes, 8.5 GB), then about 8 minutes per student run (5.2 GB), the same as plain fine-tuning.

All runs are also in `fine-tuning-results.md`.

**The honest teacher (Part 1b): frozen Qwen + LR, out of fold** (`kd_frozen_lr_oof_T1_a0.5`: T = 1, α = 0.5, the same seeds)

| | seed 423 | seed 1 | seed 2 | mean | ensemble of 3 |
|---|---|---|---|---|---|
| best-epoch dev accuracy (epoch kept) | 78.72 (8) | 79.08 (10) | 78.99 (9) | 78.93 | |
| **test, distilled from frozen Qwen + LR** | 74.49 | 74.90 | 74.91 | **74.77** (std 0.24) | **75.29** |
| test, distilled from LoRA Qwen (T = 2) | 72.94 | 73.58 | 72.93 | 73.15 | 74.02 |
| test, plain fine-tune (`r_lr2e-5`) | 72.84 | 73.11 | 73.45 | 73.13 | 73.93 |

**Conclusion: with a teacher that hadn't seen the training posts, distillation helps.**
- **+1.64 points on the 3-seed mean.** Every distilled seed beats every plain seed. Macro-F1 rose by 1.56 (73.43 vs. 71.87).
- **Ensembles: +1.36** (75.29 vs. 73.93). 303 test posts are right only for the distilled one, 203 only for the plain one: McNemar p = 1e-5. By the decision rule: distillation helps.
- **Fraction of the gap closed:**
  - 36% of the gap to this teacher (77.72 test, fitted on the same 9,987 posts);
  - 30% of the original gap to the LoRA Qwen (1.64 / 5.55).
- **The single model now beats the published RoBERTa-large fine-tune** (74.1, Edwards & Camacho-Collados; `fine-tuning-results.md`, Conclusions point 2), at a third of the size.
- **The less accurate teacher taught better.**
  - Frozen Qwen + LR scores 77.72 on test against LoRA Qwen's 78.68.
  - But its soft labels on the training posts carry real uncertainty: a median 10.8% of the probability off the true class, against 0.006%. Its confidence also matches its accuracy.
  - That's consistent with Cho & Hariharan 2019 ("more accurate teachers often don't make good teachers"). It doesn't isolate accuracy, though: the two teachers also differ in memorization, calibration and input length (`../../Distillation-lessons.md`, "Why LoRA Qwen taught worse").
- **The student learned from the teacher, not just from softer labels:**
  - It picked the teacher's class on 82.2% of the test posts, against 78.2% for the plain fine-tune.
  - Its dev loss kept falling for all 10 epochs (to about 0.70) instead of rising to about 1.4.
  - Dev understated the gain this time: +1.08 at the best epoch, against +1.64 on test.
**Controls (Part 1c)**, the same α, T and seeds:

| Test | seed 423 | seed 1 | seed 2 | mean (gain) | ensemble | agreement with frozen Qwen + LR |
|---|---|---|---|---|---|---|
| plain fine-tune | 72.84 | 73.11 | 73.45 | 73.13 | 73.93 | 78.15 |
| distilled from frozen Qwen + LR, out of fold | 74.49 | 74.90 | 74.91 | 74.77 (+1.64) | 75.29 | 82.20 |
| `label_smoothing` (ε = 0.307, no teacher) | 72.81 | 73.15 | 73.31 | 73.09 (−0.04) | 74.19 | 78.68 |
| `frozen_lr_insample` | 74.37 | 75.05 | 74.53 | 74.65 (+1.52) | 75.36 | 81.49 |

- **The gain is the teacher's information, not softer targets.** Label smoothing with the same average softness did nothing: 212 vs. 193 posts against plain, McNemar p = 0.37. The out-of-fold teacher beat it: 265 vs. 184, p = 0.0002.
- **Out-of-fold labels weren't needed for this teacher.** In-sample and out-of-fold tie: 82 vs. 87 posts, p = 0.76. This LR is strongly regularized (88.0% right on its own training posts), so unlike LoRA Qwen (98.2%) it hadn't memorized them.
**A stronger teacher (Part 1d): frozen Qwen + MLP, out of fold, temperature-calibrated** (`kd_frozen_mlp_oof_calibrated_T1_a0.5`)
- **The teacher:** 0.4–0.5 points more accurate than the LR teacher (test 78.22 vs. 77.72), and as calibrated after dividing its logits by 0.53. The two teachers agree on 93.8% of the test posts.
- **The student:** test 73.76, 74.87, 74.33, mean **74.32 (+1.19)**; ensemble 75.03.
- **It taught no better than the LR.** The ensembles differ on 102 vs. 121 posts (p = 0.23); against plain fine-tuning it's 294 vs. 213 (p = 0.0004). Dev had favored it slightly (79.26 vs. 78.93 at the best epoch); test didn't.

- **The outputs printed above are from the last run** (`frozen_mlp_oof_calibrated`, seed 2), since every run re-executes the notebook.

**Next steps (not built yet).** The honest teacher above was the first step. The next gives the student new inputs to imitate the teacher on; the one after gives it 2,560 numbers per post to learn from instead of 20:
- **More transfer inputs:** the parts of long posts beyond their first 512 tokens, or crops of posts, labeled by the teacher. The student would then imitate the teacher on text it has no labels for.
- **Distill the representation, not just the scores:** train RoBERTa's `<s>` (through a 768 → 2,560 layer) to match the frozen Qwen vectors already cached in `../frozen/qwen3_4b_2048_instrB_last_20ng.npz`.
- **Tune $\alpha$:** Hinton found a lower weight on the hard part worked best.